In [ ]:
import random
import copy
import math
import time

COLORS = ['Red', 'Blue', 'Green', 'Yellow']
VALUES = [str(i) for i in range(10)] + ['Skip']

class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def __repr__(self):
        return f"{self.color} {self.value}"

    def __eq__(self, other):
        if isinstance(other, Card):
            return self.color == other.color and self.value == other.value
        return False

    def __hash__(self):
        return hash((self.color, self.value))

def generate_full_deck():
    """Generates a simplified UNO deck. 1 of each 0-9 and Skip per color (44 cards)."""
    deck = []
    for color in COLORS:
        for value in VALUES:
            deck.append(Card(color, value))
    return deck

class GameState:
    def __init__(self, player_hands, top_card, deck, current_turn=0):
        self.player_hands = player_hands  # List of 3 lists (for each player)
        self.top_card = top_card
        self.deck = deck  # List of remaining cards
        self.current_turn = current_turn

    def get_prompt_state_representation(self, ai_player_index):
        """Returns the state representation requested in the prompt."""
        opp1 = (ai_player_index + 1) % 3
        opp2 = (ai_player_index + 2) % 3
        return {
            'ai_cards': self.player_hands[ai_player_index],
            'opponent1_cards': self.player_hands[opp1],
            'opponent2_cards': self.player_hands[opp2],
            'top_card': self.top_card,
            'deck': self.deck
        }

    def is_terminal(self):
        """Rule 4: Player wins when cards = 0."""
        for hand in self.player_hands:
            if len(hand) == 0:
                return True
        return False

    def get_winner(self):
        for i, hand in enumerate(self.player_hands):
            if len(hand) == 0:
                return i
        return -1

    def clone(self):
        return GameState(
            [list(hand) for hand in self.player_hands],
            self.top_card,
            list(self.deck),
            self.current_turn
        )

def get_valid_moves(hand, top_card):
    """
    Rule 1: Valid Move
    Same color OR same number.
    Returns a list of Valid Card objects.
    """
    valid_moves = []
    for card in hand:
        if card.color == top_card.color or card.value == top_card.value:
            valid_moves.append(card)
    return valid_moves

def apply_move(state: GameState, move):
    """
    Rule 4: State Transition.
    If move is a Card, it acts as playing that card.
    If move is 'Draw', it means drawing a card.
    Note: In chance nodes, 'Draw' gets expanded to specific card draws.
    This function handles standard playing or deterministic draws.
    Returns the new state.
    """
    new_state = state.clone()
    p_idx = new_state.current_turn

    if move == 'Draw':
        # Rule 2: Drawing Card. If no valid card, player must draw 1.
        if len(new_state.deck) > 0:
            drawn = new_state.deck.pop(0)
            new_state.player_hands[p_idx].append(drawn)
        # Advance turn
        new_state.current_turn = (new_state.current_turn + 1) % 3
    else:
        # Playing a card
        # Remove card from hand
        try:
            new_state.player_hands[p_idx].remove(move)
        except ValueError:
            pass # Card already not in hand (shouldn't happen in valid simulation)

        new_state.top_card = move

        # Rule 3: Skip Card
        if move.value == 'Skip':
            new_state.current_turn = (new_state.current_turn + 2) % 3
        else:
            new_state.current_turn = (new_state.current_turn + 1) % 3

    return new_state

def apply_chance_draw(state: GameState, drawn_card: Card):
    """
    Applies the outcome of a Chance Node for Expectimax.
    Draws a *specific* drawn_card from the deck.
    """
    new_state = state.clone()
    p_idx = new_state.current_turn

    # Remove this specific card from deck, if possible (to maintain correct probabilities deep in tree)
    try:
        new_state.deck.remove(drawn_card)
    except ValueError:
        pass

    new_state.player_hands[p_idx].append(drawn_card)
    new_state.current_turn = (new_state.current_turn + 1) % 3

    return new_state

def setup_game():
    deck = generate_full_deck()
    random.shuffle(deck)

    player_hands = [[], [], []]
    for _ in range(5): # 5 cards per player
        for i in range(3):
            player_hands[i].append(deck.pop(0))

    # Initial top card must not be a skip for simplicity (or we can just let it be)
    top_card = deck.pop(0)
    while top_card.value == 'Skip':
        deck.append(top_card)
        random.shuffle(deck)
        top_card = deck.pop(0)

    return GameState(player_hands, top_card, deck, current_turn=0)